# Publish the per-source shards and the CL-0 probe set

Two artifacts v0.3.0 needs and does not have. CPU only, no GPU quota.

**1. The CL-0 probe shard.** `kalia-v020` publishes a checkpoint and two log
CSVs, but no `val.bin` -- so the forgetting probe had nothing to measure and
Step 0 could not run at all. This is a **slice of the training corpus**, not a
held-out set: the ledger exists to detect forgetting of what was learned, so the
measuring set has to be data the model was trained on. Frozen once published,
which is what makes drift comparable across updates.

**2. The four per-source shards.** `mix_bins.py` pre-blends them into one
`train.bin` and discards the originals, so Step 4's runtime mixture cannot run
however much quota is available. Re-derived from the prep output.

Published to the model repo, which is already public and licence-audited. The
shards are derived from the same filtered sources v0.2.0 was trained on, so no
new licence question arises -- and none of this redistributes raw third-party
text, only the same uint16 token stream the model already saw.

In [ ]:
import glob, hashlib, json, os, shutil

work = '/kaggle/working/kalia'
if not os.path.exists(work):
    hits = sorted(glob.glob('/kaggle/input/**/mix_bins.py', recursive=True))
    assert hits, 'attach kalia-code-dev'
    shutil.copytree(os.path.dirname(hits[0]), work)
os.chdir(work)

bins = {os.path.basename(p): p for p in glob.glob('/kaggle/input/**/*.bin', recursive=True)}
for name, p in sorted(bins.items()):
    print(f'{name:34s} {os.path.getsize(p)/1e6:9.1f} MB')
print('\nCPU kernel, no GPU requested.')

In [ ]:
# Locate the pre-blended training corpus and the per-source shards. mix_bins.py
# wrote a single train.bin; if the originals survived we skip the rebuild.
train_bins = [p for n, p in bins.items() if n.startswith('train')]
source_bins = {n: p for n, p in bins.items()
               if n in ('smollm_fineweb_edu.bin', 'tinystories.bin',
                        'smollm_cosmopedia_v2.bin', 'python.bin')}
print('pre-blended train:', train_bins)
print('per-source survivors:', sorted(source_bins))
assert train_bins, 'no train.bin in the attached prep output'

In [ ]:
# 1. The CL-0 probe shard: a frozen prefix of the training stream.
PROBE_TOKENS = 1000000
src = train_bins[0]
total = os.path.getsize(src) // 2   # uint16
take = min(PROBE_TOKENS, total)
with open(src, 'rb') as fh:
    head = fh.read(take * 2)
probe = '/kaggle/working/probe/val_forget.bin'
os.makedirs(os.path.dirname(probe), exist_ok=True)
open(probe, 'wb').write(head)
print(f'probe shard: {take:,} tokens ({take*2/1e6:.1f} MB) of {total:,}')
print('  taken from the TRAINING stream on purpose: CL-0 measures forgetting')
print('  of what was learned, so a held-out set would measure nothing.')
digest = hashlib.sha256(head).hexdigest()
print('  sha256:', digest)

In [ ]:
# 2. Per-source shards. If they survived the mix, copy them; otherwise rebuild
# from the prep output's per-source token files.
import numpy as np

RATIOS = {'fineweb': 0.60, 'tinystories': 0.20, 'cosmopedia': 0.15, 'python': 0.05}
PER_SOURCE_FILES = {
    'fineweb': ['smollm_fineweb_edu.bin'],
    'tinystories': ['tinystories.bin'],
    'cosmopedia': ['smollm_cosmopedia_v2.bin', 'cosmopedia.bin'],
    'python': ['python.bin', 'python_filtered.bin'],
}
out_dir = '/kaggle/working/sources'
os.makedirs(out_dir, exist_ok=True)
manifest = {}
for name, cands in PER_SOURCE_FILES.items():
    hit = next((bins[c] for c in cands if c in bins), None)
    if hit is None:
        print(f'{name:12s} MISSING (looked for {cands})')
        continue
    dst = f'{out_dir}/{name}.bin'
    shutil.copyfile(hit, dst)
    n = os.path.getsize(dst) // 2
    manifest[name] = {'tokens': n, 'ratio': RATIOS[name],
                     'sha256': hashlib.sha256(open(dst,'rb').read()).hexdigest()}
    print(f'{name:12s} {n:>12,} tokens  ratio {RATIOS[name]:.2f}')
json.dump(manifest, open(f'{out_dir}/manifest.json','w'), indent=2)
print('\nratios must match the static arm of Step 4 (60/20/15/5)')

In [ ]:
REPO = 'kalia-lm/kalia-v020'
from huggingface_hub import HfApi, hf_hub_upload
import glob as _g
api = HfApi()
try:
    tok = [l.split('=', 1)[1].strip() for l in open('/kaggle/input/secrets/hf_token')
           if '=' in l and not l.startswith('#')][0]
except Exception:
    tok = os.environ.get('HF_TOKEN')
assert tok, 'no HF token available in the kernel'

uploaded = []
UPLOADS = [(probe, 'corpus/probe_val.bin')]
UPLOADS += [(f'{out_dir}/{n}.bin', f'corpus/{n}.bin') for n in RATIOS]
UPLOADS += [(f'{out_dir}/manifest.json', 'corpus/manifest.json')]
for path, dest in UPLOADS:
    hf_hub_upload(REPO, path, dest, repo_type='model', token=tok)
    uploaded.append((dest, os.path.getsize(path)))
    print(f'uploaded {dest:34s} {os.path.getsize(path)/1e6:8.1f} MB')
print(f'\n{len(uploaded)} files published to {REPO}')
print('Step 0 can now run; Step 4 now has the shards its runtime mixture needs.')